# Feature Engineering


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

df = pd.read_parquet("../data/processed/fraud_ecommerce.parquet")

target = "class"
X = df.drop(columns=[target])
y = df[target]

# Identify columns
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Remove obvious IDs that shouldn't be modeled as raw
drop_cols = [c for c in ["user_id", "device_id", "ip_address"] if c in X.columns]
numeric_features = [c for c in numeric_features if c not in drop_cols]
categorical_features = [c for c in categorical_features if c not in drop_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train class distribution:")
print(y_train.value_counts())
print("Test class distribution:")
print(y_test.value_counts())


Train class distribution:
class
0    109568
1     11321
Name: count, dtype: int64
Test class distribution:
class
0    27393
1     2830
Name: count, dtype: int64


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

cc = pd.read_parquet("../data/processed/creditcard.parquet")

Xc = cc.drop(columns=["Class"])
yc = cc["Class"]

num_cols = Xc.columns.tolist()

# Optional: scale only Time/Amount (common approach)
scale_cols = [c for c in ["Time", "Amount"] if c in Xc.columns]
passthrough_cols = [c for c in Xc.columns if c not in scale_cols]

preprocess_cc = ColumnTransformer(
    transformers=[
        ("scale", StandardScaler(), scale_cols),
        ("pass", "passthrough", passthrough_cols),
    ]
)


In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("Before SMOTE (train):")
print(y_train.value_counts())

smote = SMOTE(random_state=42)

# Fit/transform ONLY train
X_train_transformed = preprocess.fit_transform(X_train)
X_res, y_res = smote.fit_resample(X_train_transformed, y_train)

print("After SMOTE (train):")
print(pd.Series(y_res).value_counts())


ImportError: cannot import name '_is_pandas_df' from 'sklearn.utils.validation' (c:\Users\Hp\Downloads\fraud-detection\.venv\Lib\site-packages\sklearn\utils\validation.py)

In [ ]:
print("Numeric features scaled:", numeric_features[:5], "...")
print("Categorical features encoded:", categorical_features[:5], "...")
print("SMOTE applied only on training data ✔")
